# Loss Function vs Cost Function


While both terms are often used interchangeably in machine learning, they have distinct mathematical scopes when optimizing a model.

## 1. Loss Function (Element-wise Error)
A **Loss Function** measures the error of a **single data point**. It calculates how far off a single prediction ($\hat{y}$) is from its true target value ($y$).

* **Scope:** Individual (per-instance).
* **Purpose:** Evaluates model performance on a single training example.
* **Notation Example (MSE Loss):** $$L(y, \hat{y}) = (y - \hat{y})^2$$

## 2. Cost Function (Aggregate Error)
A **Cost Function** measures the total error across the **entire training dataset**. It is typically calculated as the average (mean) of all individual losses, often including an extra penalty term for regularization.

* **Scope:** Global (entire dataset).
* **Purpose:** The actual objective function minimized by optimization algorithms like Gradient Descent during training.
* **Notation Example (MSE Cost):** $$J(w, b) = \frac{1}{n} \sum_{i=1}^{n} L(y^{(i)}, \hat{y}^{(i)})$$

**Bascially loss function is the error for individual data sample, but cost function is the error for all the samples in the entire dataset (bascially this is the formula for MSE).**


## Quick Summary Table

| Feature | Loss Function | Cost Function |
| :--- | :--- | :--- |
| **Data Scope** | One single training example | The entire training dataset |
| **Mathematical Role** | Component of the cost function | The final average optimization target |
| **Common Terms** | Error Function, Criterion | Objective Function, Target Function |

---

# Regression Loss Functions: MSE vs. MAE vs. Huber Loss

Choosing the right loss function is critical because it dictates how your optimization algorithm (like Gradient Descent) reacts to errors and handles outliers in your dataset.


## 1. MSE (Mean Squared Error)
MSE calculates the average of the squared differences between predicted values ($\hat{y}$) and actual values ($y$).

### **Formula**
$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y^{(i)} - \hat{y}^{(i)})^2$$

### **Pros**
* **Smooth Optimization:** The mathematical curve is a clean parabola. It is perfectly differentiable everywhere, which means gradient descent can smoothly slow down and converge precisely at the global minimum without oscillating.
* **Biased Toward Large Errors:** It acts as an excellent choice if your system considers large errors mathematically unacceptable (e.g., safety-critical systems).

### **Cons**
* **Highly Vulnerable to Outliers:** Because errors are squared, a single extreme outlier can dominate the total cost value. This forces the model's trend line to warp toward the anomaly, destroying accuracy for the rest of the normal data points.


## 2. MAE (Mean Absolute Error)
MAE calculates the average of the absolute differences between predicted values ($\hat{y}$) and actual values ($y$).

### **Formula**
$$MAE = \frac{1}{n} \sum_{i=1}^{n} |y^{(i)} - \hat{y}^{(i)}|$$

### **Pros**
* **Outlier Robustness:** It handles all errors on a strictly linear scale. An error of 100 is worth exactly 100, not 10,000. This keeps extreme anomalies from dominating and tilting the model's structural trend line.
* **Intuitive Metrics:** The resulting cost matches the physical units of your target variable directly (e.g., error measured directly in dollars or meters).

### **Cons**
* **Optimization Instability:** The mathematical curve forms a sharp "V" shape. Its derivative is discontinuous at zero, meaning the gradient remains at a constant velocity all the way down. When approaching the minimum, gradient descent will frequently overshoot and bounce back and forth (oscillate) instead of settling down smoothly.


## 3. Why Huber Loss is Critical in This Context

Huber Loss serves as a vital bridge designed to solve the structural vulnerabilities of both MSE and MAE. It acts as a **hybrid function** that dynamically shifts its mathematical behavior based on the size of the calculated error, using an outlier threshold parameter ($\delta$).


### **The Hybrid Formula**
$$L_{\delta}(y, \hat{y}) = \begin{cases} \frac{1}{2}(y - \hat{y})^2 & \text{for } |y - \hat{y}| \le \delta \quad \text{(MSE Behavior)} \\ \delta \cdot \left(|y - \hat{y}| - \frac{1}{2}\delta\right) & \text{for } |y - \hat{y}| > \delta \quad \text{(MAE Behavior)} \end{cases}$$

### **The Resolution of the Conflict**
1. **At the Bottom (Small Errors $\le \delta$):** It adopts quadratic **MSE behavior**. This ensures the base of the curve remains perfectly smooth and curved, allowing gradient descent to decelerate cleanly and lock onto the exact global minimum without bouncing.
2. **At the Edges (Large Errors $> \delta$):** It switches automatically to linear **MAE behavior**. This instantly strips away the compounding power of extreme outliers, ensuring they are penalized linearly rather than exponentially.

### **Summary Table**

| Metric / Feature | MSE | MAE | Huber Loss |
| :--- | :--- | :--- | :--- |
| **Error Scale** | Quadratic (Squared) | Linear (Absolute) | Quadratic near zero, Linear at edges |
| **Outlier Sensitivity** | Extremely High | Low (Robust) | Low (Robust, ignores extreme spikes) |
| **Gradient Behavior** | Decreases to 0 smoothly | Remains constant (Fixed step) | Decreases to 0 smoothly |
| **Convergence** | Stable and precise | Prone to oscillation near zero | Stable and precise |


## Huberb Loss (a balance of both MSE and MAE)

### The Regression Catch-22
* **MSE (Mean Squared Error):** Smooth and easily differentiable near zero, but squares errors, making it highly volatile and sensitive to outliers.
* **MAE (Mean Absolute Error):** Robust against outliers, but its derivative is discontinuous at zero (a sharp "V" shape), causing optimization gradients to stall or oscillate violently.

### The Hybrid Solution: Huber Loss Formula
Huber Loss acts like MSE when the error is small, but switches smoothly to MAE when the error crosses a threshold ($\delta$), neutralizing outlier distortion while maintaining clean optimization gradients.

$$L_{\delta}(y, \hat{y}) = \begin{cases} \frac{1}{2}(y - \hat{y})^2 & \text{for } |y - \hat{y}| \le \delta \\ \delta \cdot \left(|y - \hat{y}| - \frac{1}{2}\delta\right) & \text{for } |y - \hat{y}| > \delta \end{cases}$$

Where:
* $y$ is the actual true value.
* $\hat{y}$ is the predicted value.
* $\delta$ (Delta) is the threshold boundary distinguishing an inlier from an outlier.

### How Huber Loss Detects Outliers
Huber Loss evaluates data points dynamically on every training iteration. It calculates the absolute residual $|y - \hat{y}|$:
1. If the absolute error is **$\le \delta$**, it tags the point as an **inlier** and processes it quadratically (MSE behavior) for precise tuning.
2. If the absolute error is **$> \delta$**, it flags the point as an **outlier** and switches to a linear penalty (MAE behavior), protecting the structural trend line from tilting.

---

## Methodologies for Calculating $\delta$

There is no specific method to calculate delta, instead we have 3 different methods to get $\delta$ (delta) depending on the problem statement. <br>
Data scientists compute $\delta$ systematically using three primary methods rather than guessing flat arbitrary figures:

### **Method A: The Standard Deviation Formula**
Best used when your data is relatively uniform with sporadic, random noise spikes.
$$\delta = k \times \sigma$$
* $\sigma$ is the standard deviation of your baseline residuals.
* $k$ is a scaling constant, typically set to **$1.35$** (which mathematically ensures that the top ~20% of extreme errors are handled as outliers).

### **Method B: The Interquartile Range (IQR) Formula**
Best used when your raw training data is heavily contaminated with severe, extreme outliers that distort regular mean and standard deviation metrics.
$$\delta = 1.5 \times \text{IQR}$$
* $\text{IQR} = Q_3 - Q_1$ (The middle 50% range of your error distribution). This replicates the exact mathematical boundary limits used to establish whiskers on a box-and-whisker plot.

### **Method C: Dynamic Hyperparameter Optimization**
Used when optimizing models for precise performance thresholds (e.g., Kaggle competitions).
* **Process:** Run a Grid Search or Random Search across a specified array of decimals (e.g., `[0.1, 0.5, 1.0, 2.0]`).
* **Evaluation:** Run k-fold cross-validation to select the exact delta value that minimizes validation set loss.